# Gale Crater dust devil activity heuristic: multi-sol pipeline in one notebook

Per-pixel **heuristic score** (0 to 1) of dust devil activity on the 100 m THEMIS grid, for a list of sols. Not a calibrated probability and not a detection.

Stages, run top to bottom in one kernel:

0. Mastcam dust opacity table (`mastcam_tau_per_sol.csv`)
1. Per-pixel feature table (`gale_dust_devil_features_100m.csv` + `gale_dust_devil_meta.json`), needs rasterio and the 20 m rasters
2. Shared reference from `REF_SOL` (sol 1850), cached in `dust_devil_reference_sol1850.json`
3. Per sol: score maps (`.npz` + 2-band GeoTIFF), rover check table, overlay previews shaded in three fixed levels (low / moderate / high)
4. Per run: summary CSV and an animated GIF on the hillshade

All sols are scored against **one** reference (sol 1850), so a dustier or lower-sun sol shows less activity and the highlighted share varies by sol. Sol 1850 (Ls 76, tau 0.44 measured that sol) sits at the median of the reference statistic over the 263 sols with at least 40 REMS bins, so about half the sols are sunnier and half dimmer. The maps use a fixed grid of 96 quarter-hour LTST bins; each sol's REMS bins are used only for the rover check.

Three runs, each with its own summary CSV and GIF, named by sol range:
- `SOLS_CLEAR`: sols around the reference sol 1850 (clear season, Ls 74-81).
- `SOLS_DUSTY`: sols 799-804, the dusty season (Ls 228-231, tau about 1.5). Expect little highlighted area: that is the point of the shared reference.
- `SOLS_CLIM_CHECK`: sols whose tau falls back to the Ls climatology, to exercise and label that path.

Stages 0 and 1 are skipped when their output already exists (`"auto"`); set a flag to `True` to force a rebuild, `False` to skip. Stage 1 only needs a re-run if an input raster or the bounding box changes. The reference is rebuilt with `RUN_REFERENCE = True`; loading a cached reference built with different settings fails.

In [1]:
from pathlib import Path
import gc, json, re
import numpy as np, pandas as pd

PROC = Path("../processed_data")
OUT = Path("../gale_tiles/out_tiles")

# ---------------- which stages to run: True = always, False = never, "auto" = only if the output is missing ----------------
RUN_TAU_TABLE = "auto"
RUN_FEATURES  = "auto"
RUN_REFERENCE = "auto"   # shared reference; True rebuilds it, "auto" loads the cache (and checks its settings)

# ---------------- sols (each list is one run with its own summary CSV and GIF; [] skips it) ----------------
SOLS_CLEAR = [1830, 1840, 1845, 1850, 1860, 1865]      # clear season around the reference sol (sols with >= MIN_BINS bins)
SOLS_DUSTY = [799, 800, 801, 802, 803, 804]            # dusty season, tau about 1.5 (Mastcam)
SOLS_CLIM_CHECK = [21, 22, 23, 26, 1770, 1775, 1860]   # tau from the Ls climatology (before Mastcam / Mastcam gaps)
MIN_BINS = 40            # a sol needs at least this many 15-minute REMS bins (max 96)
REF_SOL = 1850           # every sol is scored against this sol's reference proxy and threshold (median sol; user decision Nov 2)
DEMO_SOL = 800           # stage 1 only: the rover pixel written to meta.json (stages 2-4 take the rover from each sol's REMS rows)

# ---------------- model settings ----------------
SLOPE_SCALE_DEG = 10.0   # score falls off for slopes steeper than this
DAY_LTST = (9.0, 16.0)   # window used for the daily-mean score
SCORE_CAP = 1.0          # cap on proxy / reference proxy (keeps the 0-1 range); None disables it
LIKELY_TOP_PCT = 10      # the fixed threshold is the top N% of REF_SOL's daily-mean score; other sols use the same cutoff
CLASS_TOP_PCTS = (50, 30, 10)   # overlay levels low / moderate / high: cuts at the top N% of REF_SOL's map (last = LIKELY_TOP_PCT)

# ---------------- dust opacity ----------------
TAU_FILES = {"880": Path("../raw_data") / "msl-tau-880-arch-3953.txt", "440": Path("../raw_data") / "msl-tau-440-arch-3953.txt"}   # the two Mastcam tables, used directly (no zip, no folder)
TAU_BAND = "880"         # "880", "440", or "mean" (average of the two bands, sols with both)
TAU_CSV = PROC / "mastcam_tau_per_sol.csv"   # written by stage 0, read by stages 2-3
MAX_TAU_GAP_SOLS = 5     # Mastcam measurements within this many sols are interpolated; otherwise the climatology below
TAU_CLIM_LS_WINDOW = 10.0          # climatology: median tau of Mastcam sols within this many degrees of Ls (wraps at 360)
TAU_CLIM_EXCLUDE = (2082, 2133)    # 2018 global dust storm sols, left out of the climatology
TAU_CLIM_MIN_SOLS = 5              # fail if fewer sols fall in the Ls window
assert TAU_BAND in ("880", "440", "mean"), TAU_BAND

# ---------------- input and output files ----------------
BINNED_CSV = PROC / "rems_binned_all_sols.csv"
TEMPLATE_TIF = PROC / "gale_ti_night_themis100m_native.tif"          # output grid (100 m THEMIS thermal inertia)
DEM_TIF, SLOPE_TIF, ASPECT_TIF = PROC/"gale_dem_20m.tif", PROC/"gale_slope_20m.tif", PROC/"gale_aspect_20m.tif"
ALB_TIFS = {y: PROC/f"gale_albedo_my{y}_native.tif" for y in (24, 25, 26)}
MIN_TI_COVERAGE = 0.99   # fail stage 1 if THEMIS covers less of the template

FEATURES_CSV = PROC / "gale_dust_devil_features_100m.csv"      # one row per 100 m pixel
META_JSON = PROC / "gale_dust_devil_meta.json"                 # template grid (shape + transform); stages 2-4 use nothing else from it
REFERENCE_JSON = PROC / f"dust_devil_reference_sol{REF_SOL}.json"
OUT_DIR = PROC           # per-sol npz / tif / rover table / overlays, per-run summary CSV and GIF

# ---------------- overlay previews and GIF ----------------
IMAGERY_PREVIEW   = OUT / "imagery_preview_z11.png"        # zoom-11 CTX imagery preview (.png/.jpg)
HILLSHADE_PREVIEW = OUT / "topo_preview_z11.png"   # zoom-11 hillshade preview (.png/.jpg)
PREVIEW_BOUNDS = None    # (west, east, south, north) in degrees if the previews cover MORE than the template; None = same as template
LEVEL_COLOURS = ("#FACC15", "#F97316", "#DC2626")   # low, moderate, high (yellow, orange, red)
ALPHA = 0.55
PANEL_W = 1100           # width of each panel in the side-by-side image, in pixels
SEPARATE_W = 2000        # width of each separate preview image, in pixels
GIF_W = 1000             # GIF frame width, in pixels
GIF_MS = 800             # GIF frame duration, in milliseconds

# ---------------- reporting ----------------
ROVER_CORR_FLAG = 0.9    # flag sols whose model-vs-observed rover correlation is below this

def want(flag, output):
    assert flag in (True, False, "auto"), f"flag must be True, False or 'auto', got {flag!r}"
    return (not Path(output).exists()) if flag == "auto" else flag

# settings the stage 2-4 functions read, under the same names
CFG = dict(REF_SOL=REF_SOL, SCORE_CAP=SCORE_CAP, SLOPE_SCALE_DEG=SLOPE_SCALE_DEG, DAY_LTST=DAY_LTST, LIKELY_TOP_PCT=LIKELY_TOP_PCT,
           CLASS_TOP_PCTS=CLASS_TOP_PCTS, LEVEL_COLOURS=LEVEL_COLOURS,
           TAU_BAND=TAU_BAND, IMAGERY_PREVIEW=IMAGERY_PREVIEW, HILLSHADE_PREVIEW=HILLSHADE_PREVIEW, PREVIEW_BOUNDS=PREVIEW_BOUNDS,
           ALPHA=ALPHA, PANEL_W=PANEL_W, SEPARATE_W=SEPARATE_W, GIF_W=GIF_W)

## Shared solar geometry (used by stages 1 and 2)

In [2]:
# ---- Solar geometry (Allison & McEwen 2000 style; good to ~0.1 deg) ----
SOL_DAYS = 1.0274912517
LANDING_JD_TT = 2456145.7 + 69.184/86400     # MSL landing 2012-08-06, sol 0 (approx.)
def ls_deg(sol):
    jd = LANDING_JD_TT + np.asarray(sol, float) * SOL_DAYS
    dj = jd - 2451545.0
    M = np.radians(19.3871 + 0.52402073 * dj)
    alpha = 270.3863 + 0.52403840 * dj
    pbs = (0.0071*np.cos(np.radians(0.985626*dj/2.2353+49.409)) + 0.0057*np.cos(np.radians(0.985626*dj/2.7543+168.173))
           + 0.0039*np.cos(np.radians(0.985626*dj/1.1177+191.837)) + 0.0037*np.cos(np.radians(0.985626*dj/15.7866+21.736))
           + 0.0021*np.cos(np.radians(0.985626*dj/2.1354+15.704)) + 0.0020*np.cos(np.radians(0.985626*dj/2.4694+95.528))
           + 0.0018*np.cos(np.radians(0.985626*dj/32.8493+49.095)))
    eoc = (10.691 + 3.0e-7*dj)*np.sin(M) + 0.623*np.sin(2*M) + 0.050*np.sin(3*M) + 0.005*np.sin(4*M) + 0.0005*np.sin(5*M) + pbs
    return (alpha + eoc) % 360

def sun_vector(ls, lat_deg, ltst_hr):
    """Unit vector to the Sun in local (East, North, Up). lat_deg may be an array."""
    dec = np.arcsin(np.sin(np.radians(25.19)) * np.sin(np.radians(ls)))
    lat, h = np.radians(lat_deg), np.radians((ltst_hr - 12.0) * 15.0)
    sin_alt = np.sin(lat)*np.sin(dec) + np.cos(lat)*np.cos(dec)*np.cos(h)
    cos_alt = np.sqrt(np.clip(1 - sin_alt**2, 0, 1))
    cos_alt_safe = np.where(cos_alt < 1e-6, 1e-6, cos_alt)
    east  = -np.cos(dec) * np.sin(h) / cos_alt_safe          # cos(alt)*sin(az) / cos(alt)
    north = (np.sin(dec) - sin_alt*np.sin(lat)) / (cos_alt_safe*np.cos(lat))
    return east*cos_alt, north*cos_alt, sin_alt

def solar_const(ls):
    nu = np.radians(ls - 251.0)                   # perihelion near Ls 251
    r = 1.52368 * (1 - 0.0934**2) / (1 + 0.0934*np.cos(nu))
    return 589.0 * (1.52368 / r)**2               # W/m2 at Mars distance

# Fixed LTST grid for the maps and the reference: the 96 quarter-hour REMS bin centres.
# The daily mean uses the 28 inside DAY_LTST (9.125..15.875); the daily max and the reference percentile use all 96.
# Each sol's own REMS bins are used only for the rover check, so REMS coverage gaps cannot change the maps.
MAP_LTST = np.arange(96) * 0.25 + 0.125

LS_REF = float(ls_deg(REF_SOL))
print(f"Ls at landing (sol 0): {float(ls_deg(0)):.1f} deg (expect ~151); Ls at reference sol {REF_SOL}: {LS_REF:.1f} deg; S0 = {solar_const(LS_REF):.0f} W/m2")

Ls at landing (sol 0): 150.7 deg (expect ~151); Ls at reference sol 1850: 76.4 deg; S0 = 493 W/m2


## Stage 0: Mastcam dust opacity per sol
Parses Lemmon's Mastcam aerosol tables into one median tau per sol. Mastcam starts at sol 33, so sols 1-32 have no value.

In [3]:
TAU_OUT = TAU_CSV

def build_tau_table():
    def read_tau(path):
        text = path.read_text()
        declared = int(re.search(r"Lines of data in this table:\s*(\d+)", text).group(1))
        lines = text.splitlines()
        start = next(i for i, l in enumerate(lines) if "Product_ID" in l) + 1
        rows = [l.split(",") for l in lines[start:] if l.strip()]
        assert all(len(r) == 5 for r in rows), f"{path.name}: a row does not have 5 fields"
        d = pd.DataFrame({"sol": [float(r[1]) for r in rows], "ls_deg": [float(r[2]) for r in rows], "tau": [float(r[3]) for r in rows],
                          "sigma": pd.to_numeric([r[4].strip() for r in rows], errors="coerce")})     # '******' -> NaN
        assert len(d) == declared, f"{path.name}: parsed {len(d)} rows but the header declares {declared}"
        n_bad = int((d.tau < 0).sum())                       # -1 = no validated data (none in the current files)
        d = d[d.tau >= 0]
        print(f"{path.name}: {len(d)} rows, sol {d.sol.min():.1f}..{d.sol.max():.1f}, tau {d.tau.min():.3f}..{d.tau.max():.3f}, dropped {n_bad} with tau = -1, {int(d.sigma.isna().sum())} with unreadable sigma")
        return d

    def per_sol(d):
        d = d.assign(sol_int=np.floor(d.sol).astype(int))
        return d.groupby("sol_int").agg(tau=("tau", "median"), n=("tau", "size"), tau_min=("tau", "min"), tau_max=("tau", "max"), ls_deg=("ls_deg", "first"))

    t880, t440 = per_sol(read_tau(TAU_FILES["880"])), per_sol(read_tau(TAU_FILES["440"]))
    if TAU_BAND == "880":
        out = t880
    elif TAU_BAND == "440":
        out = t440
    else:
        both = t880.join(t440, lsuffix="_880", rsuffix="_440", how="inner")
        out = pd.DataFrame({"tau": (both.tau_880 + both.tau_440) / 2, "n": both.n_880 + both.n_440,
                            "tau_min": both[["tau_min_880", "tau_min_440"]].min(axis=1), "tau_max": both[["tau_max_880", "tau_max_440"]].max(axis=1),
                            "ls_deg": both.ls_deg_880})
    out = out.rename_axis("sol").reset_index()
    out["band"] = TAU_BAND
    print(f"\n{len(out)} sols with tau ({TAU_BAND}); median tau {out.tau.median():.2f}, 95th percentile {out.tau.quantile(.95):.2f}, max {out.tau.max():.2f} (sol {int(out.sol[out.tau.idxmax()])})")

    # ---- coverage against the REMS sols you have ----
    rems_sols = np.sort(pd.read_csv(BINNED_CSV).sol.unique())
    dist = np.array([np.abs(out.sol.to_numpy() - s).min() for s in rems_sols])
    print(f"REMS sols: {len(rems_sols)} | tau on the same sol: {int((dist == 0).sum())} | within 5 sols: {int((dist <= 5).sum())} | within 10: {int((dist <= 10).sum())} | before the first tau sol ({int(out.sol.min())}): {int((rems_sols < out.sol.min()).sum())}")
    gaps = np.diff(out.sol.to_numpy()); print(f"gap between measured sols: median {np.median(gaps):.0f}, 90th percentile {np.percentile(gaps, 90):.0f}, max {gaps.max()} (after sol {int(out.sol.to_numpy()[gaps.argmax()])})")
    for s in (800, 1163):
        near = out.iloc[np.abs(out.sol - s).argmin()]; print(f"sol {s}: nearest measured sol {int(near.sol)}, tau {near.tau:.3f} (n={int(near.n)}), Ls {near.ls_deg:.1f}")
    out.to_csv(TAU_OUT, index=False, float_format="%.4f")
    print("wrote", TAU_OUT)

if want(RUN_TAU_TABLE, TAU_OUT):
    build_tau_table()
else:
    print("stage 0 skipped:", TAU_OUT, "exists" if TAU_OUT.exists() else "(not rebuilt)")

stage 0 skipped: ..\processed_data\mastcam_tau_per_sol.csv exists


## Stage 1: per-pixel feature table
Resamples DEM, slope, aspect (20 m, area average; aspect through sin/cos), the three albedo maps (bilinear) and THEMIS thermal inertia onto the 100 m template, and writes the CSV and `meta.json`. Needs rasterio; it is imported only here, and all arrays are freed when the stage ends.

In [4]:
def build_feature_table():
    import rasterio
    from rasterio.warp import reproject, Resampling

    with rasterio.open(TEMPLATE_TIF) as t:
        T_TF, T_CRS, T_SHAPE = t.transform, t.crs, (t.height, t.width)
        ti = t.read(1).astype("float32")
        if t.nodata is not None: ti[ti == t.nodata] = np.nan
    assert T_CRS.is_geographic, "template should be lon/lat"
    print("template", T_SHAPE, "pixel (deg):", T_TF.a, T_TF.e)
    print("template bounds lon %.4f..%.4f  lat %.4f..%.4f" % (T_TF.c, T_TF.c + T_TF.a*T_SHAPE[1], T_TF.f + T_TF.e*T_SHAPE[0], T_TF.f))
    ti_cov = np.isfinite(ti).mean()
    assert ti_cov >= MIN_TI_COVERAGE, f"THEMIS covers only {ti_cov:.1%} of the template"
    print(f"THEMIS thermal inertia coverage: {ti_cov:.2%}")

    def to_template(path, how=Resampling.average, band_fn=None):
        with rasterio.open(path) as s:
            a = s.read(1).astype("float32")
            if s.nodata is not None: a[a == s.nodata] = np.nan
            if band_fn is not None: a = band_fn(a)
            out = np.full(T_SHAPE, np.nan, "float32")
            reproject(a, out, src_transform=s.transform, src_crs=s.crs, dst_transform=T_TF, dst_crs=T_CRS,
                      resampling=how, src_nodata=np.nan, dst_nodata=np.nan)
        return out

    dem   = to_template(DEM_TIF)
    slope = to_template(SLOPE_TIF)
    asp_s = to_template(ASPECT_TIF, band_fn=lambda a: np.sin(np.radians(a)))   # average aspect as sin/cos, never raw degrees
    asp_c = to_template(ASPECT_TIF, band_fn=lambda a: np.cos(np.radians(a)))
    aspect = np.degrees(np.arctan2(asp_s, asp_c)) % 360
    alb = {y: to_template(p, Resampling.bilinear) for y, p in ALB_TIFS.items()}
    alb_mean = np.nanmean(np.stack(list(alb.values())), axis=0)

    layers = dict(elev_m=dem, slope_deg=slope, aspect_deg=aspect, ti_themis_tiu=ti,
                  albedo_my24=alb[24], albedo_my25=alb[25], albedo_my26=alb[26], albedo_mean=alb_mean)
    for n, a in layers.items():
        print(f"{n:14s} min {np.nanmin(a):9.3f}  max {np.nanmax(a):9.3f}  NaN {100*np.mean(~np.isfinite(a)):5.1f}%")

    rows, cols = np.meshgrid(np.arange(T_SHAPE[0]), np.arange(T_SHAPE[1]), indexing="ij")
    lat = T_TF.f + (rows + 0.5) * T_TF.e
    lon = T_TF.c + (cols + 0.5) * T_TF.a
    df = pd.DataFrame({"pixel_id": (rows * T_SHAPE[1] + cols).ravel().astype("int32"),
                       "row": rows.ravel().astype("int32"), "col": cols.ravel().astype("int32"),
                       "lon": lon.ravel(), "lat": lat.ravel(),
                       **{n: a.ravel() for n, a in layers.items()}})
    n_all = len(df)
    df = df.dropna(subset=list(layers)).reset_index(drop=True)
    print(f"kept {len(df):,} of {n_all:,} pixels ({len(df)/n_all:.1%}); dropped {n_all-len(df):,} with a missing layer")
    assert len(df) > 0.9 * n_all, "more than 10% of pixels lack a layer: check that the 20 m rasters cover the template"
    df = df.round({"lon": 6, "lat": 6, "elev_m": 2, "slope_deg": 3, "aspect_deg": 2, "ti_themis_tiu": 2,
                   "albedo_my24": 5, "albedo_my25": 5, "albedo_my26": 5, "albedo_mean": 5})   # %.5g would cut lon/lat to 0.01 deg
    df.to_csv(FEATURES_CSV, index=False)
    print("wrote", FEATURES_CSV, f"({FEATURES_CSV.stat().st_size/1e6:.0f} MB)")

    rems = pd.read_csv(BINNED_CSV)
    day = rems[rems.sol == DEMO_SOL].sort_values("bin").reset_index(drop=True)
    assert len(day) > 0, f"sol {DEMO_SOL} not in {BINNED_CSV.name}"
    assert day.latitude_deg.nunique() == 1 and day.longitude_deg_e.nunique() == 1, "rover moved during the demo sol"
    c, r = ~T_TF * (day.longitude_deg_e[0], day.latitude_deg[0]); r, c = int(r), int(c)   # affine returns (col, row)
    assert 0 <= r < T_SHAPE[0] and 0 <= c < T_SHAPE[1], "rover pixel is outside the template"

    meta = dict(demo_sol=DEMO_SOL, ls_deg=float(ls_deg(DEMO_SOL)), rover_lat=float(day.latitude_deg[0]), rover_lon=float(day.longitude_deg_e[0]), rover_elev_m=float(day.elevation_m[0]),
                rover_row=r, rover_col=c, rover_pixel_id=int(r*T_SHAPE[1]+c),
                n_rems_bins=int(len(day)), template_shape=list(T_SHAPE), template_transform=list(T_TF)[:6])
    META_JSON.write_text(json.dumps(meta, indent=2))
    print(json.dumps(meta, indent=2))

    # sanity: DEM at the rover pixel vs the elevation REMS reports for the rover
    dem_rover = float(dem[r, c])
    print(f"DEM at rover pixel {dem_rover:.1f} m vs REMS elevation {meta['rover_elev_m']:.1f} m (diff {dem_rover-meta['rover_elev_m']:+.1f} m)")
    print("rover-pixel row in the CSV:", int((df.pixel_id == meta["rover_pixel_id"]).sum()), "(expect 1)")

if want(RUN_FEATURES, FEATURES_CSV):
    build_feature_table()
    gc.collect()
else:
    print("stage 1 skipped:", FEATURES_CSV, "exists" if FEATURES_CSV.exists() else "(not rebuilt)")

stage 1 skipped: ..\processed_data\gale_dust_devil_features_100m.csv exists


## Stage 2a: inputs per sol
Loads the stage-1 feature table onto the template grid, the REMS bins and the tau table. Helpers give each sol its rover pixel (from that sol's REMS position, asserted static and inside the template) and its tau: Mastcam interpolated within `MAX_TAU_GAP_SOLS`, otherwise the median of Mastcam sols within `TAU_CLIM_LS_WINDOW` of the sol's Ls (other Mars years, storm sols excluded). The climatology is the only fallback, and its `tau_source` label (`climatology Ls±10, n=<k> sols`) goes into every output.

In [5]:
def load_grid(features_csv, meta_json):
    """Read the stage-1 feature table onto the template grid (see `grid` above). meta.json gives only shape and transform."""
    meta = json.loads(Path(meta_json).read_text())
    shape = tuple(meta["template_shape"])
    transform = tuple(meta["template_transform"])
    a, b, c, d, e, f = transform
    assert b == 0 and d == 0 and e < 0, "template is not a north-up lon/lat grid"
    feat = pd.read_csv(features_csv, usecols=["pixel_id", "row", "col", "elev_m", "slope_deg", "aspect_deg", "ti_themis_tiu", "albedo_mean"])
    assert feat.pixel_id.is_unique, "duplicate pixel_id in the features CSV"
    assert (feat.pixel_id == feat.row * shape[1] + feat.col).all(), "pixel_id does not match row/col for this template shape"
    assert feat.row.between(0, shape[0]-1).all() and feat.col.between(0, shape[1]-1).all(), "row/col outside the template"
    rr, cc = feat.row.to_numpy(), feat.col.to_numpy()

    def layer(column):
        out = np.full(shape, np.nan, "float32")
        out[rr, cc] = feat[column].to_numpy("float32")
        return out

    lat = (f + (np.arange(shape[0]) + 0.5) * e)[:, None] * np.ones((1, shape[1]), "float32")   # from the transform, as the notebook
    return dict(shape=shape, transform=transform, lat=lat, slope=layer("slope_deg"), aspect=layer("aspect_deg"),
                albedo=layer("albedo_mean"), ti=layer("ti_themis_tiu"), dem=layer("elev_m"))


def sols_with_min_bins(rems, min_bins):
    """Sorted list of sols in `rems` with at least `min_bins` REMS bins."""
    n = rems.groupby("sol").size()
    return sorted(int(s) for s in n.index[n >= min_bins])


def rover_pixel(day, transform, shape):
    """(row, col) of the rover for this sol from its REMS lat/lon: col = floor((lon - c)/a), row = floor((lat - f)/e).
    Asserts the rover is static within the sol and inside the template."""
    assert len(day) > 0, "no REMS rows for this sol"
    sol = int(day.sol.iloc[0])
    assert day.latitude_deg.nunique() == 1 and day.longitude_deg_e.nunique() == 1, f"rover moved during sol {sol}"
    a, _, c, _, e, f = transform
    col = int(np.floor((day.longitude_deg_e.iloc[0] - c) / a))      # col from lon, row from lat
    row = int(np.floor((day.latitude_deg.iloc[0] - f) / e))
    assert 0 <= row < shape[0] and 0 <= col < shape[1], f"sol {sol}: rover pixel (row {row}, col {col}) is outside the template {shape}"
    return row, col


def tau_for_sol(sol, ls, tau_table, max_gap_sols, clim_ls_window, clim_exclude, clim_min_sols):
    """(tau, tau_source) for one sol.
    Mastcam: np.interp over the table rows within max_gap_sols of `sol`.
    Otherwise climatology: median tau of rows with circular |ls_deg - ls| <= clim_ls_window, excluding sols in
    clim_exclude (inclusive range); asserts at least clim_min_sols rows and names the sol on failure.
    tau_source for climatology is exactly f"climatology Ls±{window:g}, n={k} sols"."""
    t = tau_table.sort_values("sol")
    assert t.sol.is_unique, "duplicate sols in the tau table"
    near = t[(t.sol - sol).abs() <= max_gap_sols]
    if len(near) > 0:
        tau = float(np.interp(sol, near.sol, near.tau))
        return tau, f"Mastcam {t.band.iloc[0]} nm, nearest {int((near.sol - sol).abs().min())} sol(s) away"
    d = (t.ls_deg - ls).abs() % 360
    d = np.minimum(d, 360 - d)
    storm = t.sol.between(clim_exclude[0], clim_exclude[1])
    sel = t[(d <= clim_ls_window) & ~storm]
    assert len(sel) >= clim_min_sols, (f"sol {sol}: no Mastcam tau within {max_gap_sols} sols and only {len(sel)} sols within "
                                       f"Ls {ls:.1f} ± {clim_ls_window:g} (need {clim_min_sols})")
    return float(sel.tau.median()), f"climatology Ls±{clim_ls_window:g}, n={len(sel)} sols"


rems = pd.read_csv(BINNED_CSV)
OK_SOLS = sols_with_min_bins(rems, MIN_BINS)
print(f"{rems.sol.nunique()} sols in {BINNED_CSV.name}; {len(OK_SOLS)} with >= {MIN_BINS} bins:", OK_SOLS)
tau_table = pd.read_csv(TAU_CSV)
assert tau_table.band.astype(str).nunique() == 1 and str(tau_table.band.iloc[0]) == str(TAU_BAND), \
    f"tau table band {tau_table.band.unique()} differs from TAU_BAND {TAU_BAND}: rebuild stage 0"
grid = load_grid(FEATURES_CSV, META_JSON)
print("grid", grid["shape"], f"valid TI pixels {np.isfinite(grid['ti']).mean():.2%}")


def day_of(s):
    d = rems[rems.sol == s].sort_values("bin").reset_index(drop=True)
    assert len(d) > 0, f"sol {s} not in {BINNED_CSV}"
    return d


def tau_of(s):
    return tau_for_sol(s, float(ls_deg(s)), tau_table, MAX_TAU_GAP_SOLS, TAU_CLIM_LS_WINDOW, TAU_CLIM_EXCLUDE, TAU_CLIM_MIN_SOLS)


for s in sorted(set([REF_SOL] + SOLS_CLEAR + SOLS_DUSTY + SOLS_CLIM_CHECK)):
    print(f"sol {s}: Ls {float(ls_deg(s)):.1f}, tau %.3f (%s)" % tau_of(s))

319 sols in rems_binned_all_sols.csv; 263 with >= 40 bins: [21, 22, 23, 26, 30, 33, 34, 35, 36, 37, 38, 40, 41, 42, 43, 44, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 90, 95, 100, 105, 110, 120, 121, 122, 123, 124, 130, 135, 155, 160, 165, 170, 366, 367, 368, 369, 370, 371, 372, 373, 374, 375, 584, 585, 590, 592, 610, 615, 620, 625, 630, 635, 640, 645, 650, 665, 709, 710, 720, 725, 730, 735, 740, 745, 760, 765, 799, 800, 801, 802, 803, 804, 815, 820, 825, 830, 835, 840, 845, 850, 855, 865, 1161, 1162, 1163, 1343, 1347, 1348, 1349, 1350, 1352, 1355, 1356, 1357, 1358, 1359, 1360, 1650, 1655, 1660, 1665, 1670, 1675, 1680, 1690, 1695, 1705, 1710, 1715, 1720, 1730, 1735, 1740, 1750, 1770, 1775, 1780, 1785, 1790, 1795, 1800, 1805, 1810, 1815, 1825, 1830, 1840, 1845, 1850, 1860, 1865, 2550, 2555, 2560, 2565, 2570, 2579, 2580, 2585, 2590, 2600, 2605, 2615, 2620, 2625, 2630, 2636, 2640, 2645, 2655, 2660, 2670, 

grid (1436, 1477) valid TI pixels 99.99%
sol 21: Ls 162.0, tau 0.793 (climatology Ls±10, n=69 sols)
sol 22: Ls 162.5, tau 0.796 (climatology Ls±10, n=68 sols)
sol 23: Ls 163.1, tau 0.800 (climatology Ls±10, n=65 sols)
sol 26: Ls 164.7, tau 0.796 (climatology Ls±10, n=66 sols)
sol 799: Ls 227.5, tau 1.517 (Mastcam 880 nm, nearest 0 sol(s) away)
sol 800: Ls 228.2, tau 1.498 (Mastcam 880 nm, nearest 1 sol(s) away)
sol 801: Ls 228.8, tau 1.478 (Mastcam 880 nm, nearest 0 sol(s) away)
sol 802: Ls 229.5, tau 1.460 (Mastcam 880 nm, nearest 1 sol(s) away)
sol 803: Ls 230.1, tau 1.442 (Mastcam 880 nm, nearest 0 sol(s) away)
sol 804: Ls 230.8, tau 1.468 (Mastcam 880 nm, nearest 0 sol(s) away)
sol 1770: Ls 40.2, tau 0.634 (climatology Ls±10, n=80 sols)
sol 1775: Ls 42.5, tau 0.631 (climatology Ls±10, n=77 sols)
sol 1830: Ls 67.4, tau 0.506 (Mastcam 880 nm, nearest 1 sol(s) away)
sol 1840: Ls 71.9, tau 0.489 (Mastcam 880 nm, nearest 4 sol(s) away)
sol 1845: Ls 74.2, tau 0.477 (Mastcam 880 nm, neare

## Stage 2b: the score
Per LTST bin, a unitless proxy for the ground-air temperature gap: absorbed sunlight on the tilted surface divided by the square root of thermal inertia, `(1 - albedo) * flux / sqrt(clip(TI, 50))`. Then `score = clip(proxy / REF_PROXY, 0, SCORE_CAP) * exp(-(slope / SLOPE_SCALE_DEG)^2)`, accumulated one bin at a time into the daily mean (9-16 LTST) and the daily max (all 96 bins); a (bins x H x W) array is never held.

The rover-fitted kelvin scale does not enter the maps (it would not cancel under a shared reference); it is kept per sol for the rover check only. All weights are assumptions, not fitted, and the score is a heuristic, not a probability.

In [6]:
def _surface(grid):
    """Per-run constants: surface normals, 1/sqrt(clip(ti)) denominator, latitude column."""
    lat = grid["lat"]
    assert np.all(lat == lat[:, :1]), "lat must vary by row only (north-up grid)"
    sl, asr = np.radians(grid["slope"]), np.radians(grid["aspect"])
    return dict(nE=np.sin(sl)*np.sin(asr), nN=np.sin(sl)*np.cos(asr), nU=np.cos(sl), lat_col=lat[:, :1],
                absorb=1 - grid["albedo"], sqrt_ti=np.sqrt(np.clip(grid["ti"], 50, None)))


def _proxy(s, ls, ltst_hr, tau):
    """Same arithmetic and dtypes as notebook cells 11-12: float32 flux, then absorbed flux / sqrt(ti)."""
    S0 = solar_const(ls)
    sE, sN, sU = sun_vector(ls, s["lat_col"], ltst_hr)              # (H, 1), broadcast over columns
    cos_i = np.clip(s["nE"]*sE + s["nN"]*sN + s["nU"]*sU, 0, None)
    up = sU > 0.02
    direct = np.where(up, S0 * cos_i * np.exp(-tau / np.clip(sU, 0.05, 1)), 0)
    diffuse = np.where(up, 0.15 * S0 * sU * (1 - np.exp(-tau)) * (1 + s["nU"]) / 2, 0)   # crude sky term
    flux = (direct + diffuse).astype("float32")
    p = s["absorb"] * flux
    p /= s["sqrt_ti"]
    return p


def slope_weight(slope, slope_scale_deg):
    """exp(-(slope / slope_scale_deg)^2), NaN slope -> 90 deg, float32."""
    return np.exp(-(np.nan_to_num(slope, nan=90) / slope_scale_deg)**2).astype("float32")


def reference_proxy(grid, ls, ltst_centers, tau):
    """nanpercentile of the proxy over all bins x all pixels, 99.5 (numpy's default linear interpolation).
    Bins that are zero everywhere (sun down) are counted, not stored: zeros are the smallest values (proxy >= 0),
    so they only shift the rank. This keeps memory at the lit bins x valid pixels."""
    q = 99.5
    s = _surface(grid)
    valid = np.isfinite(_proxy(s, ls, 12.0, tau))
    n = int(valid.sum())
    lit = []
    for hr in ltst_centers:
        p = _proxy(s, ls, hr, tau)
        assert np.array_equal(np.isfinite(p), valid), "proxy NaN pattern changes between bins"
        if np.any(p[valid] != 0):
            lit.append(hr)
    n_dark = len(ltst_centers) - len(lit)
    assert lit, "no sunlit bin: the reference is undefined"
    buf = np.empty(len(lit) * n, "float32")
    for k, hr in enumerate(lit):
        buf[k*n:(k+1)*n] = _proxy(s, ls, hr, tau)[valid]
    assert buf.min() >= 0, "negative proxy: the zero-skipping rank shift is invalid"
    j = q / 100 * (len(ltst_centers) * n - 1) - n_dark * n         # rank inside the stored (lit) values
    lo = int(np.floor(j))
    assert lo >= 0, "the percentile falls among the dark bins"
    hi = min(lo + 1, buf.size - 1)
    part = np.partition(buf, [lo, hi])
    v_lo, v_hi = float(part[lo]), float(part[hi])
    return v_lo + (j - lo) * (v_hi - v_lo)


def score_maps(grid, ls, ltst_centers, tau, ref_proxy, slope_scale_deg, day_ltst, score_cap):
    """Per bin: clip(proxy / ref_proxy, 0, score_cap) * slope_weight, accumulated one bin at a time
    (never a (bins, H, W) array). Returns dict likelihood_mean (bins with LTST in day_ltst, NaN where ti is NaN),
    likelihood_max (all bins), frac_at_cap. score_cap=None disables the cap. The maps pass MAP_LTST.
    frac_at_cap: fraction of valid pixels (finite ti) whose uncapped ratio proxy / ref_proxy reaches score_cap in at
    least one bin (any bin, not only the day window); 0 when score_cap is None. Decided by the user, Oct 30."""
    ltst_centers = np.asarray(ltst_centers, float)
    dm = (ltst_centers >= day_ltst[0]) & (ltst_centers <= day_ltst[1])
    assert dm.any(), f"no REMS bin inside the day window {day_ltst}"
    assert ref_proxy > 0, f"reference proxy must be positive, got {ref_proxy}"
    s = _surface(grid)
    flat = slope_weight(grid["slope"], slope_scale_deg)
    valid = np.isfinite(grid["ti"])
    ref = np.float32(ref_proxy)
    total = np.zeros(grid["shape"], "float64")
    mx, at_cap = None, np.zeros(grid["shape"], bool)
    for k, hr in enumerate(ltst_centers):
        ratio = _proxy(s, ls, hr, tau)
        ratio /= ref
        if score_cap is not None:
            at_cap |= ratio >= score_cap
            np.clip(ratio, 0, score_cap, out=ratio)
        else:
            np.clip(ratio, 0, None, out=ratio)
        ratio *= flat
        if dm[k]:
            total += ratio
        mx = ratio if mx is None else np.maximum(mx, ratio)
    mean = (total / dm.sum()).astype("float32")
    mean[~valid] = np.nan
    return dict(likelihood_mean=mean, likelihood_max=mx,
                frac_at_cap=float(at_cap[valid].mean()) if score_cap is not None else 0.0)


def rover_series(grid, row, col, ls, ltst_centers, tau, ref_proxy, slope_scale_deg, score_cap):
    """Per-bin proxy and score at one pixel, for the sol's own REMS bins (rover check only).
    Uses a 1 x 1 slice of the grid, so the arithmetic is the same as the full-grid scoring."""
    px = {k: v[row:row+1, col:col+1] for k, v in grid.items() if isinstance(v, np.ndarray)}
    s = _surface(px)
    flat = slope_weight(px["slope"], slope_scale_deg)
    ref = np.float32(ref_proxy)
    proxy, score = [], []
    for hr in ltst_centers:
        p = _proxy(s, ls, hr, tau)
        proxy.append(p[0, 0])
        ratio = p / ref
        ratio = np.clip(ratio, 0, score_cap) if score_cap is not None else np.clip(ratio, 0, None)
        score.append((ratio * flat)[0, 0])
    return np.array(proxy, "float32"), np.array(score, "float32")


def rover_check(proxy_rover, gap_obs_k):
    """Per-sol rover calibration: dict scale_k (proxy -> K, max/max as now) and corr (model vs observed gap).
    Uses bins with an observed gap and a positive modelled proxy (no sun -> excluded)."""
    x = np.asarray(proxy_rover, "float64")
    y = np.asarray(gap_obs_k, "float64")
    ok = np.isfinite(y) & (x > 0)
    assert ok.any(), "no usable bins at the rover pixel (need an observed gap and modelled sunlight)"
    x, y = x[ok], y[ok]
    assert x.std() > 0 and y.std() > 0, "rover check needs variation in both the model and the observed gap"
    scale = float(y.max() / x.max())
    corr = float(np.mean((x - x.mean()) * (y - y.mean())) / (x.std() * y.std()))   # no np.corrcoef: it calls BLAS
    return dict(scale_k=scale, corr=corr)

## Stage 2c: shared reference (sol 1850)
`REF_PROXY` is the 99.5th percentile of the proxy over the 96 bins x all pixels of `REF_SOL`; the fixed threshold is the 90th percentile (`100 - LIKELY_TOP_PCT`) of that sol's daily-mean score. Both are cached with the settings they depend on (including the LTST grid and `LIKELY_TOP_PCT`); loading a cache built with different settings fails.

Why sol 1850: the reference statistic varies about 2x across the record (10.4 to 23.1 over 263 sols). The first reference, sol 800, was the 12th dimmest, so 243 sols had more than 20% of pixels at `SCORE_CAP`. Sol 1850 is at the median (Mastcam tau measured that sol, rover correlation 0.97, 57 bins). Sols sunnier than it still reach the cap (`frac_at_cap` in the summary); dimmer sols, like the dusty sols 799-804, show little highlighted area.

In [7]:
def _settings(cfg):
    """Settings the reference depends on, including the LTST grid it was scored on (MAP_LTST), LIKELY_TOP_PCT,
    which sets the threshold, and CLASS_TOP_PCTS, which set the three level cuts."""
    cap = cfg["SCORE_CAP"]
    pcts = [float(p) for p in cfg["CLASS_TOP_PCTS"]]
    assert len(pcts) == 3 and all(a > b for a, b in zip(pcts, pcts[1:])), \
        f"CLASS_TOP_PCTS must be three strictly decreasing percentages (low, moderate, high), got {cfg['CLASS_TOP_PCTS']}"
    assert pcts[-1] == float(cfg["LIKELY_TOP_PCT"]), \
        f"the high level must be the threshold: CLASS_TOP_PCTS[-1] = {pcts[-1]:g} but LIKELY_TOP_PCT = {cfg['LIKELY_TOP_PCT']}"
    return dict(ref_sol=int(cfg["REF_SOL"]), tau_band=str(cfg["TAU_BAND"]), slope_scale_deg=float(cfg["SLOPE_SCALE_DEG"]),
                day_ltst=[float(cfg["DAY_LTST"][0]), float(cfg["DAY_LTST"][1])], score_cap=None if cap is None else float(cap),
                likely_top_pct=float(cfg["LIKELY_TOP_PCT"]), map_ltst=MAP_LTST.tolist(), class_top_pcts=pcts)


def _assert_same_settings(reference, cfg):
    for k, v in _settings(cfg).items():
        assert k in reference, f"reference has no {k!r} (built by an older version); rebuild it (RUN_REFERENCE = True)"
        got = list(reference[k]) if k in ("day_ltst", "map_ltst", "class_top_pcts") else reference[k]
        assert got == v, f"reference was built with {k}={got!r} but the config has {v!r}; rebuild it (RUN_REFERENCE = True)"


def build_reference(grid, day, ls, tau, tau_source, cfg):
    """Reference dict for cfg["REF_SOL"]: ref_sol, ref_proxy_p995, threshold (90th pct of its daily mean),
    class_cuts (its daily mean at the top CLASS_TOP_PCTS %, ascending; the last equals the threshold),
    tau, tau_source, tau_band, slope_scale_deg, day_ltst, score_cap. Scored on MAP_LTST; `day` only identifies the sol."""
    assert (day.sol == cfg["REF_SOL"]).all(), f"reference must be built from sol {cfg['REF_SOL']}"
    settings = _settings(cfg)
    ref_proxy = reference_proxy(grid, ls, MAP_LTST, tau)
    m = score_maps(grid, ls, MAP_LTST, tau, ref_proxy, cfg["SLOPE_SCALE_DEG"], cfg["DAY_LTST"], cfg["SCORE_CAP"])
    threshold = float(np.nanpercentile(m["likelihood_mean"], 100 - cfg["LIKELY_TOP_PCT"]))
    cuts = [float(np.nanpercentile(m["likelihood_mean"], 100 - p)) for p in settings["class_top_pcts"]]
    assert cuts[-1] == threshold and cuts == sorted(cuts), f"level cuts {cuts} inconsistent with threshold {threshold}"
    return dict(ref_proxy_p995=ref_proxy, threshold=threshold, class_cuts=cuts, tau=float(tau), tau_source=str(tau_source),
                **settings)


def score_levels(mean, cuts):
    """Level per pixel: 0 none, 1 low, 2 moderate, 3 high; a pixel is in a level when its score is >= that level's cut.
    NaN -> 0 (unshaded). uint8, same shape as `mean`."""
    assert len(cuts) == 3 and list(cuts) == sorted(cuts), f"cuts must be three ascending values, got {cuts}"
    lv = np.zeros(mean.shape, "uint8")
    for c in cuts:
        lv += np.nan_to_num(mean, nan=-np.inf) >= c
    return lv


def level_shares(mean, cuts):
    """Share of valid (finite) pixels in each level: dict frac_low, frac_moderate, frac_high."""
    v = mean[np.isfinite(mean)]
    lv = score_levels(v, cuts)
    return dict(frac_low=float(np.mean(lv == 1)), frac_moderate=float(np.mean(lv == 2)), frac_high=float(np.mean(lv == 3)))


def save_reference(path, reference):
    """Write the reference dict as JSON."""
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(json.dumps(reference, indent=2, ensure_ascii=False), encoding="utf-8")


def load_reference(path, cfg):
    """Read the reference JSON; assert ref_sol, tau_band, slope_scale_deg, day_ltst, score_cap, likely_top_pct and
    map_ltst equal cfg (and MAP_LTST)."""
    reference = json.loads(Path(path).read_text(encoding="utf-8"))
    _assert_same_settings(reference, cfg)
    return reference


if want(RUN_REFERENCE, REFERENCE_JSON):
    assert REF_SOL in OK_SOLS, f"reference sol {REF_SOL} has fewer than {MIN_BINS} REMS bins"
    reference = build_reference(grid, day_of(REF_SOL), float(ls_deg(REF_SOL)), *tau_of(REF_SOL), CFG)
    save_reference(REFERENCE_JSON, reference)
    print(f"built reference from sol {REF_SOL} -> {REFERENCE_JSON}")
else:
    reference = load_reference(REFERENCE_JSON, CFG)
    print(f"loaded reference {REFERENCE_JSON}")
gc.collect()
print(f"ref_proxy_p995 {reference['ref_proxy_p995']:.4f}, threshold {reference['threshold']:.4f}, level cuts "
      + " / ".join(f"{c:.3f}" for c in reference["class_cuts"]) + f" (top {reference['class_top_pcts']} %), "
      f"tau {reference['tau']:.3f} ({reference['tau_source']}), cap {reference['score_cap']}")

loaded reference ..\processed_data\dust_devil_reference_sol1850.json
ref_proxy_p995 16.4738, threshold 0.5903, level cuts 0.348 / 0.474 / 0.590 (top [50.0, 30.0, 10.0] %), tau 0.442 (Mastcam 880 nm, nearest 0 sol(s) away), cap 1.0


## Stage 3: one sol against the reference
Scores the sol on the fixed LTST grid, then runs the rover check on the sol's own REMS bins: the per-sol kelvin scale (max/max, so one REMS spike moves it) and the model-vs-observed correlation of the ground-air gap at the rover pixel. `dem_minus_rems_m` compares the DEM at the rover pixel with the elevation REMS reports.

In [8]:
def run_sol(sol, day, grid, ls, tau, tau_source, reference, cfg):
    """One sol against the shared reference. Returns dict: likelihood_mean, likelihood_max, rover_row, rover_col,
    rover_scale_k, rover_corr, frac_above_threshold, frac_at_cap, dem_minus_rems_m, table (per-bin rover DataFrame)."""
    assert (day.sol == sol).all(), f"day table holds sols other than {sol}"
    _assert_same_settings(reference, cfg)
    r, c = rover_pixel(day, grid["transform"], grid["shape"])
    m = score_maps(grid, ls, MAP_LTST, tau, reference["ref_proxy_p995"], cfg["SLOPE_SCALE_DEG"], cfg["DAY_LTST"], cfg["SCORE_CAP"])

    # rover check on the sol's own REMS bins; the kelvin scale never enters the maps
    proxy_rover, score_rover = rover_series(grid, r, c, ls, day.ltst_bin_center_hr.to_numpy(), tau, reference["ref_proxy_p995"],
                                            cfg["SLOPE_SCALE_DEG"], cfg["SCORE_CAP"])
    gap_obs = (day.ground_temp_k - day.air_temp_k).to_numpy()
    rc = rover_check(proxy_rover, gap_obs)
    tbl = day[["sol", "bin", "ltst_bin_center_hr", "air_temp_k", "ground_temp_k", "pressure_pa"]].copy()
    tbl["gap_obs_k"] = gap_obs
    tbl["ls_deg"], tbl["tau"] = ls, tau
    tbl["gap_model_k_rover"] = proxy_rover * np.float32(rc["scale_k"])
    tbl["score_rover"] = score_rover

    mean = m["likelihood_mean"]
    return dict(sol=int(sol), ls_deg=float(ls), tau=float(tau), tau_source=str(tau_source), n_bins=int(len(day)),
                likelihood_mean=mean, likelihood_max=m["likelihood_max"], rover_row=r, rover_col=c,
                rover_lat=float(day.latitude_deg.iloc[0]), rover_lon=float(day.longitude_deg_e.iloc[0]),
                rover_scale_k=rc["scale_k"], rover_corr=rc["corr"],
                frac_above_threshold=float(np.mean(mean[np.isfinite(mean)] >= reference["threshold"])),
                frac_at_cap=m["frac_at_cap"], dem_minus_rems_m=float(grid["dem"][r, c]) - float(day.elevation_m.iloc[0]),
                **level_shares(mean, reference["class_cuts"]), table=tbl)

## Stage 3b: output writers
Per sol: npz (both maps plus sol, Ls, tau, tau source, reference and threshold), a 2-band GeoTIFF on the template grid (band 1 daily mean, band 2 daily max), the rover table, and overlays on the CTX imagery and the hillshade (cyan ring = rover). Per run: one GIF frame per sol on the hillshade. Needs rasterio (GeoTIFF) and Pillow.

The overlays shade three fixed levels, cut at the top 50 / 30 / 10% of the reference sol's map (`CLASS_TOP_PCTS`): low (yellow), moderate (orange), high (red, the threshold). The cut values are the same for every sol, so a dusty sol shows its pattern in lighter levels instead of disappearing. The legend names score bands, not shares of the sol shown.

**What a score means.** A score of 0.59 means that, averaged over 9:00-16:00 local solar time, the pixel's surface heating (`(1 - albedo) * sunlight / sqrt(TI)`, times the slope weight) is 59% of the near-maximum heating on the reference sol. Example, a flat dark crater-floor pixel (row 834, col 905; albedo 0.14, TI 363): 219 W/m2 of average sunlight on sol 1850 gives 0.59 (high); dust on sol 800 cuts that to 174 W/m2 and the score to 0.47 (moderate). Not a probability and not a dust devil count.

In [9]:
def write_geotiff(path, res, reference, template_tif, day_ltst):
    import rasterio
    with rasterio.open(template_tif) as t:
        crs, tf, shape = t.crs, t.transform, (t.height, t.width)
    assert shape == res["likelihood_mean"].shape, "template and score maps come from different grids"
    assert crs.is_geographic, "template should be lon/lat"
    ref = reference["ref_sol"]
    profile = dict(driver="GTiff", dtype="float32", count=2, height=shape[0], width=shape[1], crs=crs, transform=tf, nodata=np.nan)
    with rasterio.open(path, "w", **profile) as dst:
        dst.write(res["likelihood_mean"].astype("float32"), 1)
        dst.set_band_description(1, f"score_mean_{day_ltst[0]:g}-{day_ltst[1]:g}LTST_shared_ref_sol{ref}")
        dst.write(res["likelihood_max"].astype("float32"), 2)
        dst.set_band_description(2, f"score_max_over_day_shared_ref_sol{ref}")


class Previews:
    """Loads each preview once, crops it to the template and caches resized copies by width."""

    def __init__(self, cfg, shape, transform):
        from PIL import Image
        Image.MAX_IMAGE_PIXELS = None
        self.H, self.W = shape
        a, _, x0, _, e, y0 = transform
        self.west, self.east, self.south, self.north = x0, x0 + a*self.W, y0 + e*self.H, y0
        self.cache, self.images = {}, {}
        for tag in ("imagery", "hillshade"):
            path = Path(cfg[f"{tag.upper()}_PREVIEW"])
            assert path.exists(), f"preview not found: {path}"
            assert path.suffix.lower() in (".png", ".jpg", ".jpeg"), f"{path.name}: export the preview as .png or .jpg"
            img = Image.open(path).convert("RGB")
            if cfg["PREVIEW_BOUNDS"] is not None:                    # crop the preview to the template extent
                bw, be, bs, bn = cfg["PREVIEW_BOUNDS"]
                assert bw <= self.west and be >= self.east and bs <= self.south and bn >= self.north, "PREVIEW_BOUNDS must contain the template"
                sx, sy = img.width / (be - bw), img.height / (bn - bs)
                img = img.crop((round((self.west-bw)*sx), round((bn-self.north)*sy), round((self.east-bw)*sx), round((bn-self.south)*sy)))
            self.images[tag] = img

    def base(self, tag, w):
        from PIL import Image
        if (tag, w) not in self.cache:
            self.cache[(tag, w)] = np.asarray(self.images[tag].resize((w, round(w * self.H / self.W)), Image.LANCZOS), dtype="float32")
        return self.cache[(tag, w)]

    def panel(self, tag, w, levels, rover_lon, rover_lat, colours, alpha):
        """Preview `tag` at width w with level k (1..3) blended in colours[k-1] at `alpha`; level 0 unshaded. Rover as a ring."""
        from PIL import Image, ImageDraw
        ph = round(w * self.H / self.W)
        lv = np.asarray(Image.fromarray(levels.astype("uint8")).resize((w, ph), Image.NEAREST))
        base = self.base(tag, w)
        out = base.copy()
        for k, col in enumerate(colours, start=1):
            sel = lv == k
            out[sel] = (base * (1 - alpha) + np.array(col, "float32") * alpha)[sel]
        im = Image.fromarray(out.round().astype("uint8"))
        d = ImageDraw.Draw(im)
        px = (rover_lon - self.west) / (self.east - self.west) * w
        py = (self.north - rover_lat) / (self.north - self.south) * ph
        rr = max(10, w // 110)
        d.ellipse([px-rr, py-rr, px+rr, py+rr], outline=(0, 0, 0), width=rr // 2 + 1)
        d.ellipse([px-rr, py-rr, px+rr, py+rr], outline=(0, 255, 255), width=max(3, rr // 4))
        return im


LEVEL_NAMES = ("Low", "Moderate", "High")


def _colours(cfg):
    assert len(cfg["LEVEL_COLOURS"]) == 3, "LEVEL_COLOURS needs one colour per level (low, moderate, high)"
    return [tuple(int(h[i:i+2], 16) for i in (1, 3, 5)) for h in cfg["LEVEL_COLOURS"]]


def _legend_items(res, reference):
    """(label, share of this sol) per level. Labels name fixed score bands, not shares of the sol shown."""
    c, p, ref = reference["class_cuts"], reference["class_top_pcts"], reference["ref_sol"]
    bands = [f"{c[0]:.3f}-{c[1]:.3f}", f"{c[1]:.3f}-{c[2]:.3f}", f">= {c[2]:.3f}"]
    shares = [res["frac_low"], res["frac_moderate"], res["frac_high"]]
    return [(f"{n} {b} (top {pc:g}% of sol {ref}): {sh:.1%} of this sol", sh) for n, b, pc, sh in zip(LEVEL_NAMES, bands, p, shares)]


def _draw_legend(d, x, y, items, colours, font, step):
    for k, ((label, _), col) in enumerate(zip(items, colours)):
        d.rectangle([x, y + k*step, x + 20, y + k*step + 14], fill=col)
        d.text((x + 28, y + k*step + 1), label, fill="black", font=font)


def write_overlays(out_dir, res, reference, previews, cfg):
    from PIL import Image, ImageDraw, ImageFont
    font = ImageFont.load_default()
    sol, ref = res["sol"], reference["ref_sol"]
    colours = _colours(cfg)
    levels = score_levels(res["likelihood_mean"], reference["class_cuts"])
    d0, d1 = cfg["DAY_LTST"]
    title = (f"Dust devil activity heuristic, sol {sol}, Ls {res['ls_deg']:.1f}, mean {d0:g}-{d1:g} LTST, tau={res['tau']:.2f} "
             f"({res['tau_source']}). Shared reference sol {ref}. Relative score, not a probability.")
    items = _legend_items(res, reference)
    bounds = f"lon {previews.west:.3f}..{previews.east:.3f} E, lat {previews.south:.3f}..{previews.north:.3f}"
    args = (levels, res["rover_lon"], res["rover_lat"], colours, cfg["ALPHA"])

    pw = cfg["PANEL_W"]
    ph = round(pw * previews.H / previews.W)
    head, foot, gap, step = 54, 72, 12, 20
    canvas = Image.new("RGB", (2*pw + gap, head + ph + foot), "white")
    for k, tag in enumerate(("imagery", "hillshade")):
        canvas.paste(previews.panel(tag, pw, *args), (k*(pw + gap), head))
    d = ImageDraw.Draw(canvas)
    d.text((8, 8), title, fill="black", font=font)
    d.text((8, 28), f"Left: CTX imagery   Right: hillshade   |   {bounds}", fill="black", font=font)
    _draw_legend(d, 8, head + ph + 10, items, colours, font, step)
    canvas.save(Path(out_dir) / f"likelihood_overlay_sol{sol}.png")

    for tag, name in (("imagery", "CTX imagery"), ("hillshade", "Hillshade")):
        im = previews.panel(tag, cfg["SEPARATE_W"], *args)
        head, foot = 56, 80
        sheet = Image.new("RGB", (im.width, head + im.height + foot), "white")
        sheet.paste(im, (0, head))
        d = ImageDraw.Draw(sheet)
        d.text((8, 8), f"{name}: {title}", fill="black", font=font)
        d.text((8, 30), bounds, fill="black", font=font)
        _draw_legend(d, 8, head + im.height + 12, items, colours, font, step)
        sheet.save(Path(out_dir) / f"likelihood_overlay_sol{sol}_{tag}.png")


def gif_frame(res, reference, previews, cfg):
    from PIL import Image, ImageDraw, ImageFont
    font = ImageFont.load_default()
    colours = _colours(cfg)
    levels = score_levels(res["likelihood_mean"], reference["class_cuts"])
    im = previews.panel("hillshade", cfg["GIF_W"], levels, res["rover_lon"], res["rover_lat"], colours, cfg["ALPHA"])
    head = 92
    frame = Image.new("RGB", (im.width, head + im.height), "white")
    frame.paste(im, (0, head))
    d = ImageDraw.Draw(frame)
    d.text((8, 6), f"sol {res['sol']}   Ls {res['ls_deg']:.1f}   tau {res['tau']:.2f} ({res['tau_source']}).   "
                   f"Heuristic, not a probability.", fill="black", font=font)
    _draw_legend(d, 8, 26, _legend_items(res, reference), colours, font, 20)
    return frame


SUMMARY_COLS = ["sol", "ls_deg", "tau", "tau_source", "n_bins", "frac_above_threshold", "frac_at_cap", "rover_corr",
                "rover_scale_k", "dem_minus_rems_m", "frac_low", "frac_moderate", "frac_high"]

## Stage 4: run the sols
`run_sols` writes every per-sol output, then the run's summary CSV (`dust_devil_multisol_summary_<first>-<last>.csv`) and GIF (`dust_devil_multisol_<first>-<last>.gif`). It overwrites earlier per-sol files of the same sol (a sol in two runs gets the same files twice); the pre-multi-sol sol-800 outputs are kept in `../processed_data/baseline_sol800/`.

In [10]:
previews = Previews(CFG, grid["shape"], grid["transform"])


def run_sols(sols):
    """Score `sols` against the shared reference and write all outputs to OUT_DIR. Returns the summary DataFrame."""
    sols = sorted(int(s) for s in sols)
    assert len(sols) == len(set(sols)) and len(sols) > 0, f"sols must be a non-empty list of distinct sols: {sols}"
    short = [s for s in sols if s not in OK_SOLS]
    assert not short, f"sols with fewer than {MIN_BINS} REMS bins: {short}"
    out = Path(OUT_DIR)
    rows, frames = [], []
    for s in sols:
        ls = float(ls_deg(s))
        tau, src = tau_of(s)
        res = run_sol(s, day_of(s), grid, ls, tau, src, reference, CFG)
        np.savez_compressed(out / f"gale_dust_devil_likelihood_sol{s}.npz",
                            likelihood_mean=res["likelihood_mean"].astype("float32"), likelihood_max=res["likelihood_max"].astype("float32"),
                            sol=s, ls_deg=ls, tau=tau, tau_source=src, ref_sol=reference["ref_sol"], ref_proxy=reference["ref_proxy_p995"],
                            threshold=reference["threshold"], score_cap=np.nan if SCORE_CAP is None else float(SCORE_CAP),
                            day_ltst=np.array(DAY_LTST), rover_row=res["rover_row"], rover_col=res["rover_col"])
        write_geotiff(out / f"gale_dust_devil_likelihood_sol{s}.tif", res, reference, TEMPLATE_TIF, DAY_LTST)
        res["table"].to_csv(out / f"rover_sol{s}_table.csv", index=False)
        write_overlays(out, res, reference, previews, CFG)
        frames.append(gif_frame(res, reference, previews, CFG))
        rows.append({k: res[k] for k in SUMMARY_COLS})
        print(f"sol {s}: Ls {ls:.1f}, tau {tau:.3f} ({src}), {res['n_bins']} bins, above threshold {res['frac_above_threshold']:.1%}, "
              f"at cap {res['frac_at_cap']:.1%}, rover (row {res['rover_row']}, col {res['rover_col']}) corr {res['rover_corr']:.2f}, "
              f"scale {res['rover_scale_k']:.3f} K, DEM - REMS {res['dem_minus_rems_m']:+.1f} m")
        del res
        gc.collect()

    summary = pd.DataFrame(rows, columns=SUMMARY_COLS)
    summary.to_csv(out / f"dust_devil_multisol_summary_{sols[0]}-{sols[-1]}.csv", index=False)
    frames[0].save(out / f"dust_devil_multisol_{sols[0]}-{sols[-1]}.gif", save_all=True, append_images=frames[1:],
                   duration=GIF_MS, loop=0)
    print(f"wrote summary and GIF for sols {sols[0]}-{sols[-1]} ({len(frames)} frames)")
    return summary


def flag(summary):
    """Print the summary with the sols that need a second look: low rover correlation or climatology tau."""
    pd.set_option("display.width", 250)
    print(summary.to_string(index=False, float_format=lambda x: f"{x:.4g}"))
    low = summary[summary.rover_corr < ROVER_CORR_FLAG]
    clim = summary[summary.tau_source.str.startswith("climatology")]
    print(f"\nrover correlation below {ROVER_CORR_FLAG}:", low[["sol", "rover_corr"]].to_dict("records") or "none")
    print("climatology tau:", clim[["sol", "tau", "tau_source"]].to_dict("records") or "none")

### Clear season around the reference sol
Sols 1830-1865 (Ls 74-81). Sol 1860 has no Mastcam tau within 5 sols and uses the climatology (labelled).

In [11]:
if SOLS_CLEAR:
    summary_clear = run_sols(SOLS_CLEAR)
    flag(summary_clear)

sol 1830: Ls 67.4, tau 0.506 (Mastcam 880 nm, nearest 1 sol(s) away), 48 bins, above threshold 6.4%, at cap 3.2%, rover (row 341, col 370) corr 0.84, scale 2.243 K, DEM - REMS +11.1 m


sol 1840: Ls 71.9, tau 0.489 (Mastcam 880 nm, nearest 4 sol(s) away), 45 bins, above threshold 6.8%, at cap 3.5%, rover (row 341, col 370) corr 0.96, scale 2.122 K, DEM - REMS +6.4 m


sol 1845: Ls 74.2, tau 0.477 (Mastcam 880 nm, nearest 1 sol(s) away), 49 bins, above threshold 7.4%, at cap 4.0%, rover (row 341, col 370) corr 0.94, scale 2.122 K, DEM - REMS +6.4 m


sol 1850: Ls 76.4, tau 0.442 (Mastcam 880 nm, nearest 0 sol(s) away), 57 bins, above threshold 10.0%, at cap 6.3%, rover (row 342, col 370) corr 0.97, scale 2.704 K, DEM - REMS +19.8 m


sol 1860: Ls 80.9, tau 0.476 (climatology Ls±10, n=82 sols), 62 bins, above threshold 6.8%, at cap 3.6%, rover (row 342, col 370) corr 0.94, scale 2.034 K, DEM - REMS +19.8 m


sol 1865: Ls 83.1, tau 0.450 (Mastcam 880 nm, nearest 1 sol(s) away), 52 bins, above threshold 8.9%, at cap 5.4%, rover (row 342, col 370) corr 0.91, scale 1.823 K, DEM - REMS +20.3 m


wrote summary and GIF for sols 1830-1865 (6 frames)
 sol  ls_deg    tau                            tau_source  n_bins  frac_above_threshold  frac_at_cap  rover_corr  rover_scale_k  dem_minus_rems_m  frac_low  frac_moderate  frac_high
1830   67.43 0.5055 Mastcam 880 nm, nearest 1 sol(s) away      48                 0.064       0.0315      0.8369          2.243             11.07    0.2219         0.1934      0.064
1840   71.91 0.4888 Mastcam 880 nm, nearest 4 sol(s) away      45               0.06798      0.03502      0.9566          2.122              6.43    0.2193         0.1946    0.06798
1845   74.16 0.4767 Mastcam 880 nm, nearest 1 sol(s) away      49               0.07391      0.03969      0.9364          2.122              6.44    0.2154         0.1962    0.07391
1850    76.4  0.442 Mastcam 880 nm, nearest 0 sol(s) away      57                   0.1       0.0629      0.9676          2.704             19.77       0.2            0.2        0.1
1860   80.89 0.4765          climatolo

### Dusty season: sols 799-804
Tau about 1.5 (Mastcam). Scored against the clear reference, these sols show mostly low and moderate levels and almost no high: the map says the dusty season is quieter, while the pattern stays visible.

In [12]:
if SOLS_DUSTY:
    summary_dusty = run_sols(SOLS_DUSTY)
    flag(summary_dusty)

sol 799: Ls 227.5, tau 1.517 (Mastcam 880 nm, nearest 0 sol(s) away), 58 bins, above threshold 0.2%, at cap 0.0%, rover (row 304, col 372) corr 0.98, scale 3.702 K, DEM - REMS +24.2 m


sol 800: Ls 228.2, tau 1.498 (Mastcam 880 nm, nearest 1 sol(s) away), 60 bins, above threshold 0.3%, at cap 0.0%, rover (row 304, col 372) corr 0.97, scale 3.475 K, DEM - REMS +24.2 m


sol 801: Ls 228.8, tau 1.478 (Mastcam 880 nm, nearest 0 sol(s) away), 58 bins, above threshold 0.3%, at cap 0.1%, rover (row 304, col 372) corr 0.94, scale 3.734 K, DEM - REMS +24.2 m


sol 802: Ls 229.5, tau 1.460 (Mastcam 880 nm, nearest 1 sol(s) away), 58 bins, above threshold 0.4%, at cap 0.1%, rover (row 304, col 372) corr 0.97, scale 3.560 K, DEM - REMS +24.2 m


sol 803: Ls 230.1, tau 1.442 (Mastcam 880 nm, nearest 0 sol(s) away), 60 bins, above threshold 0.4%, at cap 0.1%, rover (row 304, col 372) corr 0.95, scale 3.731 K, DEM - REMS +23.7 m


sol 804: Ls 230.8, tau 1.468 (Mastcam 880 nm, nearest 0 sol(s) away), 56 bins, above threshold 0.3%, at cap 0.1%, rover (row 304, col 372) corr 0.97, scale 3.577 K, DEM - REMS +23.7 m


wrote summary and GIF for sols 799-804 (6 frames)
 sol  ls_deg   tau                            tau_source  n_bins  frac_above_threshold  frac_at_cap  rover_corr  rover_scale_k  dem_minus_rems_m  frac_low  frac_moderate  frac_high
 799   227.5 1.517 Mastcam 880 nm, nearest 0 sol(s) away      58               0.00209    0.0002815      0.9789          3.702             24.17    0.2731        0.06699    0.00209
 800   228.2 1.498 Mastcam 880 nm, nearest 1 sol(s) away      60              0.002531     0.000413      0.9699          3.475             24.17    0.2737         0.0743   0.002531
 801   228.8 1.478 Mastcam 880 nm, nearest 0 sol(s) away      58              0.003077    0.0005738      0.9428          3.734             24.17    0.2736          0.082   0.003077
 802   229.5  1.46 Mastcam 880 nm, nearest 1 sol(s) away      58              0.003699    0.0007464      0.9744           3.56             24.17    0.2731        0.08929   0.003699
 803   230.1 1.442 Mastcam 880 nm, nearest 0 

### Climatology tau
Sols before Mastcam starts (sol 33) and in Mastcam gaps. Their tau comes from other Mars years at the same Ls, so read them as a check of the fallback path, not as observed dust conditions. The run spans years and rover positions, so its GIF frames are not a time series.

In [13]:
if SOLS_CLIM_CHECK:
    summary_clim = run_sols(SOLS_CLIM_CHECK)
    flag(summary_clim)

sol 21: Ls 162.0, tau 0.793 (climatology Ls±10, n=69 sols), 40 bins, above threshold 16.1%, at cap 16.4%, rover (row 261, col 410) corr 0.80, scale 3.013 K, DEM - REMS +3.3 m


sol 22: Ls 162.5, tau 0.796 (climatology Ls±10, n=68 sols), 41 bins, above threshold 16.1%, at cap 16.5%, rover (row 261, col 410) corr 0.90, scale 3.194 K, DEM - REMS +4.2 m


sol 23: Ls 163.1, tau 0.800 (climatology Ls±10, n=65 sols), 41 bins, above threshold 16.2%, at cap 16.6%, rover (row 261, col 410) corr 0.94, scale 3.160 K, DEM - REMS +4.2 m


sol 26: Ls 164.7, tau 0.796 (climatology Ls±10, n=66 sols), 43 bins, above threshold 17.3%, at cap 19.7%, rover (row 261, col 411) corr 0.97, scale 2.762 K, DEM - REMS -1.5 m


sol 1770: Ls 40.2, tau 0.634 (climatology Ls±10, n=80 sols), 51 bins, above threshold 6.6%, at cap 2.7%, rover (row 339, col 367) corr 0.95, scale 3.346 K, DEM - REMS +27.9 m


sol 1775: Ls 42.5, tau 0.631 (climatology Ls±10, n=77 sols), 54 bins, above threshold 5.9%, at cap 2.3%, rover (row 339, col 367) corr 0.92, scale 3.366 K, DEM - REMS +27.9 m


sol 1860: Ls 80.9, tau 0.476 (climatology Ls±10, n=82 sols), 62 bins, above threshold 6.8%, at cap 3.6%, rover (row 342, col 370) corr 0.94, scale 2.034 K, DEM - REMS +19.8 m


wrote summary and GIF for sols 21-1860 (7 frames)
 sol  ls_deg    tau                   tau_source  n_bins  frac_above_threshold  frac_at_cap  rover_corr  rover_scale_k  dem_minus_rems_m  frac_low  frac_moderate  frac_high
  21     162  0.793 climatology Ls±10, n=69 sols      40                0.1608       0.1638      0.8009          3.013              3.33     0.169         0.1977     0.1608
  22   162.5 0.7965 climatology Ls±10, n=68 sols      41                0.1612        0.165      0.9025          3.194              4.23    0.1688         0.1976     0.1612
  23   163.1    0.8 climatology Ls±10, n=65 sols      41                0.1617       0.1662      0.9446           3.16              4.23    0.1685         0.1976     0.1617
  26   164.7 0.7965 climatology Ls±10, n=66 sols      43                 0.173       0.1969      0.9746          2.762             -1.53    0.1636         0.1959      0.173
1770   40.23  0.634 climatology Ls±10, n=80 sols      51               0.06602      0

## Checks
1. The threshold and the three level cuts are the top `CLASS_TOP_PCTS` of the reference sol's own daily-mean map (exact), and that sol has about 20 / 20 / 10% of pixels in the low / moderate / high levels.
2. Dusty sols score below the clear reference: sols 799-804 should have a smaller share above the threshold than sol 1850.
3. The arithmetic itself is checked by the test gates (`tests/`), including the regression against the single-sol sol-800 output (`baseline_sol800/`) and Gate 11 for a reference sol other than 800.

In [14]:
zr = np.load(OUT_DIR / f"gale_dust_devil_likelihood_sol{REF_SOL}.npz")
thr_check = float(np.nanpercentile(zr["likelihood_mean"], 100 - LIKELY_TOP_PCT))
v = zr["likelihood_mean"][np.isfinite(zr["likelihood_mean"])]
print(f"threshold {reference['threshold']!r} vs p{100 - LIKELY_TOP_PCT} of sol {REF_SOL}'s map {thr_check!r}: equal {thr_check == reference['threshold']}")
print(f"sol {REF_SOL}: {np.mean(v >= reference['threshold']):.2%} of pixels above the threshold (expect about {LIKELY_TOP_PCT}%)")
assert thr_check == reference["threshold"], "threshold does not match the reference sol's map"
cuts_check = [float(np.nanpercentile(zr["likelihood_mean"], 100 - p)) for p in CLASS_TOP_PCTS]
assert cuts_check == reference["class_cuts"], "level cuts do not match the reference sol's map"
print(f"level cuts {reference['class_cuts']} match; sol {REF_SOL} shares", level_shares(zr["likelihood_mean"], reference["class_cuts"]))

runs = {name: globals()[f"summary_{name}"] for name in ("clear", "dusty", "clim") if f"summary_{name}" in globals()}
for name, s in runs.items():
    print(f"{name:5s} run: above threshold {s.frac_above_threshold.min():.1%}..{s.frac_above_threshold.max():.1%}, "
          f"low {s.frac_low.min():.1%}..{s.frac_low.max():.1%}, moderate {s.frac_moderate.min():.1%}..{s.frac_moderate.max():.1%}, "
          f"at cap {s.frac_at_cap.min():.1%}..{s.frac_at_cap.max():.1%}, tau {s.tau.min():.2f}..{s.tau.max():.2f}")

threshold 0.5903030037879944 vs p90 of sol 1850's map 0.5903030037879944: equal True
sol 1850: 10.00% of pixels above the threshold (expect about 10%)


level cuts [0.3476359248161316, 0.4741957485675812, 0.5903030037879944] match; sol 1850 shares {'frac_low': 0.2000005658178142, 'frac_moderate': 0.19999962278812386, 'frac_high': 0.1000002829089071}
clear run: above threshold 6.4%..10.0%, low 20.0%..22.2%, moderate 19.3%..20.0%, at cap 3.2%..6.3%, tau 0.44..0.51
dusty run: above threshold 0.2%..0.4%, low 27.2%..27.4%, moderate 6.7%..9.7%, at cap 0.0%..0.1%, tau 1.44..1.52
clim  run: above threshold 5.9%..17.3%, low 16.4%..22.5%, moderate 19.2%..19.8%, at cap 2.3%..19.7%, tau 0.48..0.80
